# Packages import

In [8]:
import requests
from bs4 import BeautifulSoup


# Ceneo Scraper

1. Provide url of the products opinion page

In [3]:
product_code = "133893145"
url = f"https://www.ceneo.pl/{product_code}#tab=reviews"

2. Send request to provided url

In [ ]:
response = requests.get(url)
print(response.status_code)

3. Fetch product name

In [ ]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

In [ ]:
product_name = page_dom.select_one("h1").get_text()
print(product_name)
print(type(product_name))

4. Fetch all opinions

In [ ]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

5. Parse opinions to extract required data

In [ ]:
structured_opinions = []

for opinion in opinions:
    opinion_id = opinion["data-entry-id"]

    author = opinion.select_one("span.user-post__author-name").get_text(strip=True)

    recommendation_tag = opinion.select_one("span.user-post__author-recomendation > em")
    recommendation = recommendation_tag.get_text(strip=True) if recommendation_tag else "Brak"

    stars = opinion.select_one("span.user-post__score-count").get_text(strip=True)

    content = opinion.select_one("div.user-post__text").get_text(strip=True)

    features = opinion.select("div.review-feature__title")
    advantages = []
    disadvantages = []

    for feature in features:
        label = feature.get_text(strip=True)
        items = [item.get_text(strip=True) for item in feature.find_next_sibling("div").select("div.review-feature__item")]
        if "Zalety" in label:
            advantages = items
        elif "Wady" in label:
            disadvantages = items

    useful = opinion.select_one("button.vote-yes")["data-total-vote"]
    unuseful = opinion.select_one("button.vote-no")["data-total-vote"]

    dates = opinion.select("span.user-post__published > time")
    pub_date = dates[0]["datetime"] if len(dates) > 0 else None
    pur_date = dates[1]["datetime"] if len(dates) > 1 else None

    structured_opinions.append({
        "id": opinion_id,
        "author": author,
        "recommendation": recommendation,
        "stars": stars,
        "content": content,
        "advantages": advantages,
        "disadvantages": disadvantages,
        "useful_count": useful,
        "unuseful_count": unuseful,
        "pub_date": pub_date,
        "pur_date": pur_date
    })
